# Judge a design document, step by step

This notebook runs the pipeline on one document so you can see what each stage produces: the units found mechanically, the units the model finds, the checks, the scores, and the files a run writes. The last section judges every document in a folder and shows the metric-by-document table.

Run from `evals/`: `uv run jupyter lab`, then open this file. Model stages need `ANTHROPIC_API_KEY` in `evals/.env`; `DESIGN_JUDGE_MODEL` there picks the judge (the `--model` argument overrides it). Without a key the model stages are skipped.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from design import mechanical, prompts
from design.case import case_from_docs, load_case, load_docs
from design.client import Client, default_model, has_credentials
from design.extraction import extract_llm
from design.judge import run_case

EVALS = Path.cwd().resolve()
if EVALS.name == 'notebooks':
    EVALS = EVALS.parent
DESIGNS = EVALS / 'private' / 'designs'           # internal documents (git-ignored); fall back to the public fixtures
if not DESIGNS.is_dir():
    DESIGNS = EVALS / 'fixtures' / 'designs'
FLOW = EVALS / 'private' / 'runs' / 'notebook'    # where runs are written (git-ignored)
VARIANT = 'baseline'                               # baseline = designs from the current skill; v1, v2, ... after a skill change
VOTES = 1                                          # 3 takes a majority per check; costs three times as much
METRICS = None                                     # None = every rubric that applies; or a list of ids, e.g. ['paragraph-earns-place']
pd.set_option('display.max_colwidth', 140)
LIVE = has_credentials()
print('model calls enabled:', LIVE, '| judge model:', default_model())
sorted(p.name for p in DESIGNS.glob('*.md'))

## 1. Pick a document

Set `DOC` to a file in the folder above (or any path). A case built from a document alone has no task and no fact sheet; `cases/*.json` can add those.

In [ ]:
DOC = sorted(DESIGNS.glob('*.md'))[0]   # or any path
case = case_from_docs([str(DOC)])
doc = load_docs(case)[0]
print(case.id, '|', doc.kind, '|', doc.name, '|', len(doc.text), 'chars')
print(doc.header)

## 2. Mechanical units

Paragraphs (list items and table rows included), headings, code fences and mermaid diagrams, with ordinal ids and character offsets. Near-duplicate sentence pairs and hedge words are found here too.

In [ ]:
ex = mechanical.extract(doc.text)
print('covariates:', json.dumps(ex.covariates))
print('hedges:', [(h['unit'], h['word']) for h in ex.spans['hedges']])
print('duplicate pairs:', [(u.id, u.meta['a']['unit'], u.meta['b']['unit'], u.meta['similarity']) for u in ex.by_kind('duplicate_pair')])
pd.DataFrame([{'id': u.id, 'kind': u.kind, 'form': u.meta.get('form'), 'sentences': u.meta.get('sentences'), 'words': u.meta.get('words'), 'lines': u.meta.get('lines'), 'text': u.text[:100].replace('\n', ' ')} for u in ex.units])

In [ ]:
[{k: v for k, v in g.meta.items() if k not in ('fence_start', 'fence_end')} for g in ex.by_kind('diagram')]

## 3. What the judge sees

Every model call gets the same stable prefix (the reviewer stance in `design/prompts/system.md`, then the document with `[pNN]` labels), so the document is served from the prompt cache after the first call. The question comes last.

In [ ]:
print(prompts.system_prompt()[:500], '...')
print()
print(prompts.marked_document(doc.text, ex)[:800], '...')

## 4. Units the model finds

Decisions, components, claims about the existing system, relational passages, the overview span and detailed parts. Each quote is anchored in the text and the ids are assigned by position; anything the model quotes that is not in the document is dropped (`dropped` in the report). Look at this list first: every per-unit score depends on it.

In [ ]:
if LIVE:
    client = Client(cache_dir=FLOW / VARIANT / 'private' / case.id / 'cache')
    report = extract_llm(client, doc, ex)
    print(report, '| model:', client.model, '| usage:', client.usage.to_dict(client.model))
    print('overview span:', ex.spans['overview'], '| open questions:', ex.spans['open_questions'])
    display(pd.DataFrame([{'id': u.id, 'kind': u.kind, 'summary / name': u.meta.get('summary') or u.meta.get('name') or u.meta.get('kind'), 'quote': u.text[:120].replace('\n', ' ')} for u in ex.units if u.kind in ('decision', 'component', 'claim', 'relational_passage', 'detailed_part')]))
else:
    print('skipped: no credentials')

## 5. Judge the document

Runs every rubric that applies (or `METRICS`), with `VOTES` votes per check, and writes the run into `FLOW/VARIANT/`. The extraction is cached per run directory, so re-running this cell after a rubric edit does not pay for extraction again.

In [ ]:
if LIVE:
    res = run_case(case, flow=FLOW, variant=VARIANT, rep=0, metrics=METRICS, votes=VOTES)
    print(res.usage)
else:
    res = run_case(case, flow=FLOW, variant=VARIANT, rep=0, mechanical_only=True)
pd.DataFrame(res.scores)[['id', 'name', 'scoring', 'direction', 'value', 'passed', 'failed', 'unknown', 'total', 'confidence']]

The failing and undecided checks, each with the span the judge relied on. This is the list to read when deciding whether the document or the check's wording is wrong.

In [ ]:
pd.DataFrame([c.to_dict() for c in res.checks if c.verdict != 'pass'])[['metric', 'unit', 'item', 'verdict', 'votes', 'reason', 'quote']]

## 6. What was written

Runs live under `private/runs/` (git-ignored). `share/<case>/` is the only folder meant to leave your machine: the document, `units.json`, `checks.json` and `digest.md`. Paths, usage, raw model output and the cache stay in `private/`.

In [ ]:
for p in sorted((FLOW / VARIANT).rglob('*')):
    if p.is_file() and 'cache' not in p.parts and 'raw' not in p.parts:
        print(p.relative_to(FLOW))
print()
print((res.share_dir / 'digest.md').read_text())

## 7. Every document in the folder

Judges each `*.md` in `DESIGNS` into the same flow and variant (skipping the one above if it already ran), then shows metric by document. Cost scales with document length: roughly 25 calls per document per vote.

In [ ]:
RUN_ALL = False   # set True to judge every document in DESIGNS
if RUN_ALL and LIVE:
    for p in sorted(DESIGNS.glob('*.md')):
        c = case_from_docs([str(p)])
        if (FLOW / VARIANT / 'traces' / f'{c.id}_rep0.json').exists():
            print('already judged:', c.id); continue
        r = run_case(c, flow=FLOW, variant=VARIANT, rep=0, metrics=METRICS, votes=VOTES)
        print(c.id, r.usage)
rows = []
rf = FLOW / VARIANT / 'results.jsonl'
if rf.exists():
    for ln in rf.read_text().splitlines():
        r = json.loads(ln)
        rows.append({'case': r['prompt_id'], 'rep': r['rep'], **r['grade'], **{f'perf.{k}': v for k, v in r['perf'].items() if k in ('words', 'code_lines', 'diagrams', 'cost_usd')}})
pd.DataFrame(rows).set_index(['case', 'rep']).T if rows else 'no runs yet'